# Laboratorio 2 — Tomografía de Resistividad Eléctrica (ERT)
**Anexo: código del estudio de sensibilidad**

Maestría en Ciencias – Geofísica · Universidad Nacional de Colombia
Milton Daniel Díaz Barrera · Daniela Liseth Saballet Franco

Entorno: Python 3.11, PyGIMLi 1.5.5, NumPy 1.26 (conda, canales `gimli` y
`conda-forge`). La esfera 3D (sección 3) requiere además `tetgen`.

Los resultados pueden variar levemente entre ejecuciones (del orden de 1 punto porcentual en la recuperación y 0.05 en χ²).

## 0. Configuración

In [ ]:
# necesario en VS Code para ver las figuras como imagen
%matplotlib inline
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from matplotlib.path import Path

import pygimli as pg
import pygimli.meshtools as mt
from pygimli.physics import ert

pg.setLogLevel(40)          # solo errores en el registro de PyGIMLi
print("PyGIMLi", pg.__version__, "| NumPy", np.__version__)

In [ ]:
# Parametros globales (identicos en todos los casos)
OUTDIR = "resultados_ert"
os.makedirs(OUTDIR, exist_ok=True)

N_ELEC = 21                # electrodos
X_ELEC = (-15.0, 15.0)     # extremos de la linea (espaciamiento 1.5 m)
ESQUEMA = "dd"             # dipolo-dipolo

NOISE_LEVEL = 0.01         # ruido relativo (fraccion: 1 %)
NOISE_ABS = 1e-6           # ruido absoluto (V)
SEED = 1337                # semilla fija del ruido

LAMBDA = 20                # regularizacion por defecto
QUALITY = 34               # calidad de la malla directa (angulo minimo)
VERBOSE_INV = False

# Marcadores de region
M_CAPA1, M_CAPA2, M_BASAM, M_OBJ = 1, 2, 3, 4

## Funciones comunes

In [ ]:
# Objetos del modelo con capas (seccion ~7 m2, centro a ~3 m)
OBJETOS = {
    "circulo":     dict(tipo="circulo", centro=(0.0, -3.0), radio=1.5),
    "dique_vert":  dict(tipo="rect", x=(-0.85, 0.85), z=(-5.3, -1.3)),
    "tabular_hor": dict(tipo="rect", x=(-3.5, 3.5), z=(-3.6, -2.6)),
}


def crear_objeto(nombre):
    """Poligono del objeto para insertarlo en la geometria (PLC)."""
    o = OBJETOS[nombre]
    if o["tipo"] == "circulo":
        return mt.createCircle(pos=list(o["centro"]), radius=o["radio"], nSegments=48,
                               marker=M_OBJ, boundaryMarker=10, area=0.05)
    x0, x1 = sorted(o["x"])
    z0, z1 = sorted(o["z"])
    return mt.createRectangle(start=[x0, z1], end=[x1, z0], marker=M_OBJ,
                              boundaryMarker=10, area=0.05)


def dentro_del_objeto(x, z, nombre):
    o = OBJETOS[nombre]
    if o["tipo"] == "circulo":
        cx, cz = o["centro"]
        return (x - cx) ** 2 + (z - cz) ** 2 <= o["radio"] ** 2
    x0, x1 = sorted(o["x"])
    z0, z1 = sorted(o["z"])
    return (x0 <= x <= x1) and (z0 <= z <= z1)


def marker_en(x, z, z1, z2, geometria):
    """Region del modelo verdadero en el punto (x, z)."""
    if dentro_del_objeto(x, z, geometria):
        return M_OBJ
    if z > z1:
        return M_CAPA1
    if z > z2:
        return M_CAPA2
    return M_BASAM


def media_geom(v):
    v = np.asarray(v)
    return float(np.exp(np.mean(np.log(v)))) if v.size else float("nan")


def rugosidad_modelo(mgr):
    """sqrt(Phi_m): termino de suavidad que minimiza la inversion."""
    return float(np.sqrt(mgr.inv.phiModel()))


def revisar_inversion(modelPD, chi2, nombre):
    """Marca como no valida una inversion que no itero (modelo homogeneo o chi2 alto)."""
    problemas = []
    if np.std(np.log(modelPD)) < 1e-3:
        problemas.append("modelo invertido homogeneo (no itero)")
    if chi2 > 50:
        problemas.append("chi2 = %.0f" % chi2)
    if problemas:
        print("  RESULTADO NO VALIDO en %s: %s" % (nombre, "; ".join(problemas)))
        return False
    return True

In [ ]:
def correr_caso(nombre, z1=-1.0, z2=-7.0, rho_capa1=100., rho_capa2=50.,
                rho_basam=200., rho_obj=500., geometria="circulo", lam=LAMBDA,
                mostrar=True, devolver_objetos=False):
    """Modelo verdadero -> simulacion con ruido -> inversion -> metricas y figura."""
    # 1. Modelo verdadero
    world = mt.createWorld(start=[-50, 0], end=[50, -50], layers=[z1, z2],
                           worldMarker=True)
    geom = world + crear_objeto(geometria)
    rhomap = [[M_CAPA1, rho_capa1], [M_CAPA2, rho_capa2],
              [M_BASAM, rho_basam], [M_OBJ, rho_obj]]

    # 2. Esquema de medicion; nodos extra en cada electrodo (singularidad)
    scheme = ert.createData(elecs=np.linspace(X_ELEC[0], X_ELEC[1], N_ELEC),
                            schemeName=ESQUEMA)
    for p in scheme.sensors():
        geom.createNode(p)
        geom.createNode(p - [0, 0.1])
    mesh = mt.createMesh(geom, quality=QUALITY)

    # 3. Simulacion + ruido; se descartan rhoa negativas
    data = ert.simulate(mesh, scheme=scheme, res=rhomap, noiseLevel=NOISE_LEVEL,
                        noiseAbs=NOISE_ABS, seed=SEED, verbose=False)
    n_total = data.size()
    data.remove(data["rhoa"] < 0)

    # 4. Inversion (malla de inversion automatica del ERTManager)
    mgr = ert.ERTManager(data)
    model = mgr.invert(data, lam=lam, verbose=VERBOSE_INV)
    modelPD = np.array(mgr.paraModel(model))
    chi2 = float(mgr.inv.chi2())
    rrms = float(mgr.inv.relrms())
    valido = revisar_inversion(modelPD, chi2, nombre)

    # 5. Metricas: region verdadera de cada celda invertida
    pd_ = mgr.paraDomain
    xc = np.array([c.center()[0] for c in pd_.cells()])
    zc = np.array([c.center()[1] for c in pd_.cells()])
    mk = np.array([marker_en(x, z, z1, z2, geometria) for x, z in zip(xc, zc)])
    rho_rec_obj = media_geom(modelPD[mk == M_OBJ])
    rho_rec_enc = media_geom(modelPD[mk == M_CAPA2])
    c_real = rho_obj / rho_capa2
    c_rec = rho_rec_obj / rho_rec_enc
    recup = np.log(c_rec) / np.log(c_real) if abs(np.log(c_real)) > 1e-9 else np.nan

    # centroide de la anomalia: capa 2, |x| <= 8 m, mismo signo que el objeto
    dev = np.log(modelPD / rho_rec_enc)
    peso = np.abs(dev)
    peso[np.sign(dev) != np.sign(np.log(c_real))] = 0.0
    peso[(zc > z1) | (zc < z2) | (np.abs(xc) > 8)] = 0.0
    peso[peso < 0.15] = 0.0
    z_rec = float(np.sum(peso * zc) / np.sum(peso)) if peso.sum() > 0 else np.nan
    z_real = float(np.mean(zc[mk == M_OBJ]))
    rug = rugosidad_modelo(mgr)

    print("%-22s chi2=%5.2f  rRMS=%4.2f%%  contraste real=%6.2f rec=%5.2f  recup=%3.0f%%"
          % (nombre, chi2, rrms, c_real, c_rec, 100 * recup))

    # 6. Figura: modelo verdadero vs invertido
    todas = [rho_capa1, rho_capa2, rho_basam, rho_obj]
    cmin, cmax = min(todas) * 0.8, max(todas) * 1.2
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 6), sharex=True, sharey=True)
    pg.show(mesh, rhomap, ax=ax1, hold=True, cMap="Spectral_r", logScale=True,
            orientation="vertical", cMin=cmin, cMax=cmax, label=pg.unit("res"))
    ax1.set_title("Modelo verdadero - %s" % nombre)
    pg.show(pd_, modelPD, ax=ax2, hold=True, cMap="Spectral_r", logScale=True,
            orientation="vertical", cMin=cmin, cMax=cmax, label=pg.unit("res"))
    ax2.set_title(r"Inversion ($\lambda$=%g, $\chi^2$=%.2f, rRMS=%.1f%%)" % (lam, chi2, rrms))
    o = OBJETOS[geometria]
    if o["tipo"] == "circulo":
        ax2.add_patch(plt.Circle(o["centro"], o["radio"], fill=False, ec="k", ls="--"))
    else:
        (x0, x1), (zz0, zz1) = sorted(o["x"]), sorted(o["z"])
        ax2.add_patch(plt.Rectangle((x0, zz0), x1 - x0, zz1 - zz0, fill=False,
                                    ec="k", ls="--"))
    for zi in (z1, z2):
        ax2.axhline(zi, c="k", ls=":", lw=0.8)
    for ax in (ax1, ax2):
        ax.set_xlim(-18, 18); ax.set_ylim(-12, 0); ax.set_ylabel("z (m)")
    ax2.set_xlabel("x (m)")
    fig.tight_layout()
    fig.savefig(os.path.join(OUTDIR, "caso_%s.png" % nombre), dpi=150)
    for n in plt.get_fignums():          # figura vacia que crea PyGIMLi
        if not plt.figure(n).axes:
            plt.close(n)
    plt.show() if mostrar else plt.close(fig)

    r = dict(caso=nombre, valido=valido, geometria=geometria, lam=lam, z1=z1, z2=z2,
             espesor_capa2=abs(z2 - z1), rho_capa1=rho_capa1, rho_capa2=rho_capa2,
             rho_basam=rho_basam, rho_obj=rho_obj, n_datos=data.size(),
             n_descartados=n_total - data.size(), chi2=round(chi2, 3),
             rrms_pct=round(rrms, 2), contraste_real=round(c_real, 3),
             contraste_recuperado=round(c_rec, 3), recuperacion_log=round(float(recup), 3),
             z_centroide_real=round(z_real, 2), z_centroide_recuperado=round(z_rec, 2),
             rugosidad=round(rug, 2))
    if devolver_objetos:
        r.update(_mesh=mesh, _scheme=scheme, _pd=pd_, _model=modelPD)
    return r


def perfil(pd_, modelo, xs, zs):
    """Valor del modelo invertido en los puntos (x, z)."""
    out = []
    for x, z in zip(xs, zs):
        c = pd_.findCell(pg.Pos(x, z))
        out.append(modelo[c.id()] if c is not None else np.nan)
    return np.array(out)


BASE = dict(z1=-1.0, z2=-7.0, rho_capa1=100., rho_capa2=50., rho_basam=200.,
            rho_obj=500., geometria="circulo", lam=20)
resultados = []

## 1. Espesor de la capa que aloja el objeto

Capa 2 de 4, 6 y 10 m (`z2` = −5, −7, −11 m). Objeto circular fijo en z = −3 m.

In [ ]:
res_A = {}
for etiqueta, z2 in [("A1_espesor_4m", -5.0), ("A2_espesor_6m", -7.0),
                     ("A3_espesor_10m", -11.0)]:
    cfg = dict(BASE); cfg.update(nombre=etiqueta, z2=z2, devolver_objetos=True)
    res_A[etiqueta] = correr_caso(**cfg)
    resultados.append({k: v for k, v in res_A[etiqueta].items() if not k.startswith("_")})

In [ ]:
# Perfiles verticales en x = 9 m (lejos del objeto) y contacto aparente
# (profundidad donde el modelo invertido cruza 100 Ohm*m)
zs = np.linspace(-0.2, -11.8, 200)
fig, axes = plt.subplots(1, 3, figsize=(13, 5), sharey=True)
for ax, (etq, r) in zip(axes, res_A.items()):
    verdadero = np.where(zs > r["z1"], r["rho_capa1"], np.where(zs > r["z2"], r["rho_capa2"],
                                                                r["rho_basam"]))
    inv = perfil(r["_pd"], r["_model"], np.full_like(zs, 9.0), zs)
    ax.semilogx(verdadero, zs, "k-", label="verdadero")
    ax.semilogx(inv, zs, "-", lw=2, label="invertido")
    cruce = np.nan
    for k in range(len(zs) - 1):
        if zs[k] < -1.5 and inv[k] < 100 <= inv[k + 1]:
            cruce = np.interp(100, [inv[k], inv[k + 1]], [zs[k], zs[k + 1]])
            break
    ax.axhline(-5.7, c="purple", ls="-.", lw=1)
    ax.set_title("%s\ncontacto real %.1f m | aparente %.1f m" % (etq, r["z2"], cruce))
    ax.set_xlabel("rho (Ohm*m)")
    ax.legend(loc="lower left")
    print("%s: contacto real %.1f m, aparente %.1f m, rho max invertido %.0f Ohm*m"
          % (etq, r["z2"], cruce, np.nanmax(inv)))
axes[0].set_ylabel("z (m)")
fig.tight_layout()
fig.savefig(os.path.join(OUTDIR, "perfiles_espesores.png"), dpi=150)
plt.show()

## 2. Resistividad del objeto

Encajante 50 Ω·m; objeto de 100 (×2), 1000 (×20) y 5 Ω·m (÷10).

In [ ]:
for etiqueta, rho_obj in [("B1_contraste_bajo_x2", 100.), ("B2_resistivo_x20", 1000.),
                          ("B3_conductivo_div10", 5.)]:
    cfg = dict(BASE); cfg.update(nombre=etiqueta, rho_obj=rho_obj)
    resultados.append(correr_caso(**cfg))

## 3. Geometrías: circular, triangular y esférica

Medio homogéneo de 50 Ω·m con objeto de 500 Ω·m (misma sección ≈ 7 m², centro a
−3 m). Circular y triangular: 2.5D. Esférico: simulación 3D (TetGen) con factor
geométrico numérico e inversión 2D. Se varía una sola cosa a la vez: número de
electrodos, λ, malla de simulación y malla de inversión.

In [ ]:
OUT_P3 = os.path.join(OUTDIR, "punto3")
os.makedirs(OUT_P3, exist_ok=True)

RHO_FONDO, RHO_OBJ3 = 50., 500.
M_FONDO3, M_OBJ3 = 1, 2

GEO_P3 = {
    "circular":   dict(dim=2, tipo="circulo", centro=(0.0, -3.0), radio=1.5),
    "triangular": dict(dim=2, tipo="poligono",
                       vertices=[(-2.0, -1.8), (2.0, -1.8), (0.0, -5.3)]),
    "esferico":   dict(dim=3, tipo="circulo", centro=(0.0, -3.0), radio=1.5),
}
NOMBRES_GEO = list(GEO_P3.keys())

# Malla de simulacion. 2D: quality, area maxima en el objeto y en todo el mundo.
# 3D: quality de tetraedros y area maxima de los triangulos de la esfera.
MALLAS_DIR = {
    "gruesa": dict(q2=30, area_obj2=0.5,  area2=0.0, q3=1.5, area_obj3=1.0),
    "media":  dict(q2=34, area_obj2=0.05, area2=0.0, q3=1.3, area_obj3=0.3),
    "fina":   dict(q2=34, area_obj2=0.01, area2=1.0, q3=1.2, area_obj3=0.1),
}
# Malla de inversion: paraDX relativo al espaciamiento; paraMaxCellSize en m2
MALLAS_INV = {
    "gruesa": dict(paraDX=1.0,  paraMaxCellSize=10.0),
    "media":  dict(paraDX=0.33, paraMaxCellSize=1.0),
    "fina":   dict(paraDX=0.2,  paraMaxCellSize=0.25),
}
PARA_DEPTH = 8.0


def esquema_p3(n_elec):
    return ert.createData(elecs=np.linspace(X_ELEC[0], X_ELEC[1], n_elec),
                          schemeName=ESQUEMA)


def objeto_plc(geo, malla):
    g, m = GEO_P3[geo], MALLAS_DIR[malla]
    if g["dim"] == 3:
        cx, cz = g["centro"]
        d = 2 * g["radio"]   # createSphere usa size (diametro); ignora radius
        return mt.createSphere(pos=[cx, 0.0, cz], size=[d, d, d], nSegments=32,
                               nRings=16, marker=M_OBJ3, boundaryMarker=10,
                               area=m["area_obj3"])
    if g["tipo"] == "circulo":
        return mt.createCircle(pos=list(g["centro"]), radius=g["radio"], nSegments=48,
                               marker=M_OBJ3, boundaryMarker=10, area=m["area_obj2"])
    return mt.createPolygon(g["vertices"], isClosed=True, marker=M_OBJ3,
                            boundaryMarker=10, area=m["area_obj2"])


def dentro_p3(x, z, geo):
    g = GEO_P3[geo]
    if g["tipo"] == "circulo":
        cx, cz = g["centro"]
        return (x - cx) ** 2 + (z - cz) ** 2 <= g["radio"] ** 2
    return Path(g["vertices"]).contains_point((x, z))


_cache_datos = {}

def simular_p3(geo, n_elec, malla_dir):
    """Datos sinteticos con ruido (cache por geometria, electrodos y malla)."""
    clave = (geo, n_elec, malla_dir)
    if clave in _cache_datos:
        return _cache_datos[clave]
    dim, m = GEO_P3[geo]["dim"], MALLAS_DIR[malla_dir]
    scheme = esquema_p3(n_elec)
    if dim == 2:
        geom = mt.createWorld(start=[-50, 0], end=[50, -50], worldMarker=True) \
            + objeto_plc(geo, malla_dir)
        for p in scheme.sensors():
            geom.createNode(p)
            geom.createNode(p - [0, 0.1])
        mesh = mt.createMesh(geom, quality=m["q2"], area=m["area2"])
        sr = True
    else:
        geom = mt.createWorld(start=[-50, -50, 0], end=[50, 50, -50], worldMarker=True) \
            + objeto_plc(geo, malla_dir)
        for p in scheme.sensors():
            geom.createNode(p)
            geom.createNode(p - [0, 0, 0.1])
        mesh = mt.createMesh(geom, quality=m["q3"])
        sr = False
        # factor geometrico numerico: k = 1/u en un medio de 1 Ohm*m
        hom = ert.simulate(mesh, scheme=scheme, res=1.0, sr=False, calcOnly=True,
                           verbose=False)
        scheme["k"] = 1.0 / hom["u"]

    # error de la malla: medio homogeneo con factor geometrico analitico
    s_an = pg.DataContainerERT(scheme)
    s_an["k"] = pg.core.geometricFactors(s_an, dim=3)
    d_hom = ert.simulate(mesh, scheme=s_an, res=1.0, sr=sr, noiseLevel=0, verbose=False)
    error_malla = 100 * float(np.max(np.abs(np.array(d_hom["rhoa"]) - 1.0)))

    rhomap = [[M_FONDO3, RHO_FONDO], [M_OBJ3, RHO_OBJ3]]
    limpio = ert.simulate(mesh, scheme=scheme, res=rhomap, sr=sr, noiseLevel=0,
                          verbose=False)
    data = ert.simulate(mesh, scheme=scheme, res=rhomap, sr=sr, noiseLevel=NOISE_LEVEL,
                        noiseAbs=NOISE_ABS, seed=SEED, verbose=False)
    data.remove(data["rhoa"] < 0)
    anomalia = 100 * float(np.max(np.abs(np.array(limpio["rhoa"]) / RHO_FONDO - 1)))
    res = dict(data=data, n_celdas_dir=mesh.cellCount(),
               error_malla_pct=round(error_malla, 2), anomalia_rhoa_pct=round(anomalia, 1))
    _cache_datos[clave] = res
    return res


def invertir_p3(geo, subpunto, variante, n_elec=21, lam=20, malla_dir="media",
                malla_inv="media"):
    """Simula (o reutiliza) los datos, invierte en 2D y calcula metricas."""
    t0 = time.time()
    sim = simular_p3(geo, n_elec, malla_dir)
    data = pg.DataContainerERT(sim["data"])
    for i in range(data.sensorCount()):                       # electrodos en (x, 0)
        data.setSensorPosition(i, [data.sensorPosition(i)[0], 0.0, 0.0])
    data["k"] = pg.core.geometricFactors(data, dim=3)          # k analitico para 2D

    imesh = ert.createInversionMesh(data, quality=33, paraDepth=PARA_DEPTH,
                                    **MALLAS_INV[malla_inv])
    mgr = ert.ERTManager(data)
    modelPD = np.array(mgr.paraModel(mgr.invert(data, mesh=imesh, lam=lam, verbose=False)))
    pd_ = mgr.paraDomain
    chi2 = float(mgr.inv.chi2())
    valido = revisar_inversion(modelPD, chi2, "%s / %s" % (geo, variante))

    xc = np.array([c.center()[0] for c in pd_.cells()])
    zc = np.array([c.center()[1] for c in pd_.cells()])
    en_obj = np.array([dentro_p3(x, z, geo) for x, z in zip(xc, zc)])
    cerca = (np.abs(xc) <= 10) & (zc >= -7)
    rec = np.log(media_geom(modelPD[en_obj]) / media_geom(modelPD[cerca & ~en_obj])) \
        / np.log(RHO_OBJ3 / RHO_FONDO)
    rho_max = float(modelPD[(np.abs(xc) <= 5) & (zc >= -7)].max())

    r = dict(geometria=geo, subpunto=subpunto, variante=variante, n_elec=n_elec, lam=lam,
             malla_dir=malla_dir, malla_inv=malla_inv, valido=valido, n_datos=data.size(),
             n_celdas_dir=sim["n_celdas_dir"], n_celdas_inv=pd_.cellCount(),
             error_malla_pct=sim["error_malla_pct"],
             anomalia_rhoa_pct=sim["anomalia_rhoa_pct"], chi2=round(chi2, 2),
             rrms_pct=round(float(mgr.inv.relrms()), 2), rho_max_rec=round(rho_max, 1),
             recuperacion_log=round(float(rec), 3), rugosidad=round(rugosidad_modelo(mgr), 2),
             tiempo_s=round(time.time() - t0, 1), _pd=pd_, _model=modelPD)
    print("  %-10s %-18s datos=%4d celdas_inv=%5d chi2=%5.2f rho_max=%6.1f recup=%3.0f%%"
          % (geo, variante, r["n_datos"], r["n_celdas_inv"], r["chi2"], r["rho_max_rec"],
             100 * r["recuperacion_log"]))
    return r


def contorno_p3(ax, geo, **kw):
    g = GEO_P3[geo]
    estilo = dict(fill=False, ec="k", ls="--", lw=1.2)
    estilo.update(kw)
    if g["tipo"] == "circulo":
        ax.add_patch(plt.Circle(g["centro"], g["radio"], **estilo))
    else:
        ax.add_patch(plt.Polygon(g["vertices"], closed=True, **estilo))


NORM_P3 = LogNorm(vmin=30, vmax=600)

def figura_p3(resultados_sub, titulo, archivo):
    """Grilla 3x3: filas = geometrias, columnas = variantes."""
    variantes = list(dict.fromkeys(r["variante"] for r in resultados_sub))
    fig, axes = plt.subplots(3, len(variantes), figsize=(4.2 * len(variantes), 7.8),
                             sharex=True, sharey=True, squeeze=False)
    for i, geo in enumerate(NOMBRES_GEO):
        for j, var in enumerate(variantes):
            ax = axes[i, j]
            r = next(x for x in resultados_sub
                     if x["geometria"] == geo and x["variante"] == var)
            pg.show(r["_pd"], r["_model"], ax=ax, hold=True, colorBar=False,
                    cMap="Spectral_r", logScale=True, cMin=30, cMax=600)
            contorno_p3(ax, geo)
            ax.set_xlim(-12, 12); ax.set_ylim(-8, 0)
            ax.set_title("%s | %s\nchi2=%.2f  rho_max=%.0f"
                         % (geo, var, r["chi2"], r["rho_max_rec"]), fontsize=9)
            ax.set_xlabel(""); ax.set_ylabel("")
    sm = plt.cm.ScalarMappable(norm=NORM_P3, cmap="Spectral_r")
    cb = fig.colorbar(sm, ax=axes, shrink=0.8, label="Resistividad (Ohm*m)")
    cb.set_ticks([30, 50, 100, 200, 500])
    cb.set_ticklabels(["30", "50", "100", "200", "500"])
    fig.suptitle(titulo, fontsize=12)
    fig.savefig(os.path.join(OUT_P3, archivo), dpi=150, bbox_inches="tight")
    plt.show()


resultados_p3 = []

In [ ]:
# Anomalia maxima en rhoa (sin ruido) del caso base
for geo in NOMBRES_GEO:
    print("%-10s anomalia maxima en rhoa: %5.1f %%"
          % (geo, simular_p3(geo, 21, "media")["anomalia_rhoa_pct"]))

### 3.1 Número de electrodos (11, 21, 41 sobre la misma línea de 30 m)

In [ ]:
sub = [invertir_p3(g, "3.1 electrodos", "%d electrodos" % n, n_elec=n)
       for g in NOMBRES_GEO for n in (11, 21, 41)]
resultados_p3 += sub
figura_p3(sub, "3.1 - Numero de electrodos (lambda = 20)", "p3_1_electrodos.png")

### 3.2 Regularizador λ (2, 20, 200)

In [ ]:
sub = [invertir_p3(g, "3.2 lambda", "lambda=%g" % l, lam=l)
       for g in NOMBRES_GEO for l in (2, 20, 200)]
resultados_p3 += sub
figura_p3(sub, "3.2 - Regularizador lambda (21 electrodos)", "p3_2_lambda.png")

### 3.3a Malla de simulación (gruesa, media, fina)

In [ ]:
sub = [invertir_p3(g, "3.3a malla simulacion", "simulacion " + m, malla_dir=m)
       for g in NOMBRES_GEO for m in ("gruesa", "media", "fina")]
resultados_p3 += sub
figura_p3(sub, "3.3a - Malla de simulacion (21 electrodos, lambda = 20)",
          "p3_3a_malla_simulacion.png")
pd.DataFrame(sub)[["geometria", "variante", "n_celdas_dir", "error_malla_pct",
                   "anomalia_rhoa_pct", "recuperacion_log"]]

### 3.3b Malla de inversión (gruesa, media, fina)

In [ ]:
sub = [invertir_p3(g, "3.3b malla inversion", "inversion " + m, malla_inv=m)
       for g in NOMBRES_GEO for m in ("gruesa", "media", "fina")]
resultados_p3 += sub
figura_p3(sub, "3.3b - Malla de inversion (mismos datos, lambda = 20)",
          "p3_3b_malla_inversion.png")

### 3.4 Resumen

In [ ]:
df3 = pd.DataFrame([{k: v for k, v in r.items() if not k.startswith("_")}
                    for r in resultados_p3])
df3.to_csv(os.path.join(OUT_P3, "metricas_punto3.csv"), index=False)
print("Casos no validos: %d de %d" % ((~df3["valido"]).sum(), len(df3)))
df3[["geometria", "subpunto", "variante", "n_datos", "n_celdas_inv", "error_malla_pct",
     "chi2", "rho_max_rec", "recuperacion_log"]]

## Complementarios (modelo con capas)

### C. Forma del objeto dentro de la capa 2

In [ ]:
for etiqueta, geo in [("C1_circulo", "circulo"), ("C2_dique_vert", "dique_vert"),
                      ("C3_tabular_hor", "tabular_hor")]:
    cfg = dict(BASE); cfg.update(nombre=etiqueta, geometria=geo)
    resultados.append(correr_caso(**cfg))

### D. Regularizador λ y curva L

In [ ]:
resultados_lam = []
for lam in [1, 5, 20, 100, 500]:
    cfg = dict(BASE); cfg.update(nombre="D_lambda_%g" % lam, lam=lam)
    r = correr_caso(**cfg)
    resultados_lam.append(r)
    resultados.append(r)

dl = pd.DataFrame(resultados_lam)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
ax1.plot(dl["rugosidad"], dl["rrms_pct"], "o-")
for _, r in dl.iterrows():
    ax1.annotate(r"$\lambda$=%g" % r["lam"], (r["rugosidad"], r["rrms_pct"]),
                 textcoords="offset points", xytext=(8, 5))
ax1.set_xlabel("Rugosidad (sqrt(Phi_m))"); ax1.set_ylabel("rRMS (%)"); ax1.set_title("Curva L")
ax2.semilogx(dl["lam"], dl["chi2"], "o-")
ax2.axhline(1.0, ls="--", c="gray")
ax2.set_xlabel(r"$\lambda$"); ax2.set_ylabel(r"$\chi^2$")
for ax in (ax1, ax2):
    ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(os.path.join(OUTDIR, "curva_L.png"), dpi=150)
plt.show()
print("lambda con chi2 mas cercano a 1:", dl.loc[(dl["chi2"] - 1).abs().idxmin(), "lam"])

## Tabla de métricas (secciones 1, 2 y complementarios)

In [ ]:
df = pd.DataFrame(resultados)
df.to_csv(os.path.join(OUTDIR, "metricas.csv"), index=False)
print("Casos no validos: %d de %d" % ((~df["valido"]).sum(), len(df)))
df[["caso", "valido", "chi2", "rrms_pct", "contraste_real", "contraste_recuperado",
    "recuperacion_log", "z_centroide_real", "z_centroide_recuperado", "rugosidad"]]